<a href="https://colab.research.google.com/github/abhiramdinesh-ops/marketing-forecast_gADS/blob/main/Python_Forecast_Gsheet_Google_Ads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =========================================
# INSTALL LIBRARIES
# =========================================
!pip install prophet psycopg2-binary gspread oauth2client pandas numpy

# =========================================
# IMPORT LIBRARIES
# =========================================
import pandas as pd
import numpy as np
import psycopg2
from prophet import Prophet
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import logging
logging.getLogger("prophet").setLevel(logging.ERROR)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)

# =========================================
# REDSHIFT CONNECTION
# =========================================
import os

REDSHIFT_HOST = os.getenv("REDSHIFT_HOST")
REDSHIFT_USER = os.getenv("REDSHIFT_USER")
REDSHIFT_PASSWORD = os.getenv("REDSHIFT_PASSWORD")
REDSHIFT_DB = os.getenv("REDSHIFT_DB")
REDSHIFT_PORT = 5439

conn = psycopg2.connect(
    host=REDSHIFT_HOST,
    port=REDSHIFT_PORT,
    dbname=REDSHIFT_DB,
    user=REDSHIFT_USER,
    password=REDSHIFT_PASSWORD
)
print("Connected to Redshift")

# =========================================
# SQL QUERY
# =========================================
query = """
WITH native_data AS (
  SELECT
    campaign_id,
    DATE(gmt_time) AS data_date,
    SUM(CASE WHEN pixel_status = 'TY' THEN 1 ELSE 0 END) AS conversions
  FROM adtopia2.native_visitors_main
  WHERE is_test = 0
  GROUP BY campaign_id, DATE(gmt_time)
),
spend_data AS (
  SELECT
    campaign_id,
    data_date,
    ad_campaign_id,
    SUM(cost / 1000000.0) AS cost
  FROM adtopia2.google_campaign_cost
  WHERE (cost / 1000000.0) > 0
  GROUP BY campaign_id, data_date, ad_campaign_id
)
SELECT
    sd.data_date,
    ac.campaign_name,
    SUM(sd.cost) AS cost,
    SUM(COALESCE(nd.conversions,0)) AS conversions
FROM spend_data sd
LEFT JOIN native_data nd
    ON sd.campaign_id = nd.campaign_id
    AND sd.data_date = nd.data_date
LEFT JOIN adtopia2.adtopia_campaigns ac
    ON sd.ad_campaign_id = ac.id
GROUP BY sd.data_date, ac.campaign_name
ORDER BY sd.data_date
"""

df = pd.read_sql(query, conn)
conn.close()
print("Rows loaded:", df.shape)

# =========================================
# DATA CLEANING
# =========================================
df['data_date'] = pd.to_datetime(df['data_date'])
df = df.dropna(subset=['campaign_name'])
df = df[df['campaign_name'].str.strip() != ""]
df = df[df['conversions'] > 0]
df['cpl'] = df['cost'] / df['conversions']

# Exclude today (partial data) — mirrors Power BI script logic
today         = df['data_date'].max()
df            = df[df['data_date'] < today]
reference_date = df['data_date'].max()

cutoff = reference_date - pd.Timedelta(days=59)
df     = df[df['data_date'] >= cutoff]
print("After cleaning:", df.shape)
print("Reference date (yesterday):", reference_date.date())

# =========================================
# DATE WINDOWS
# =========================================
last7_start = reference_date - pd.Timedelta(days=6)
prev7_start = reference_date - pd.Timedelta(days=13)
prev7_end   = reference_date - pd.Timedelta(days=7)
next7_end   = reference_date + pd.Timedelta(days=7)

# =========================================
# HELPER — RUN PROPHET ON A DAILY DATAFRAME
# Matches Power BI script exactly:
#   - IQR outlier capping
#   - log transform
#   - same Prophet params
# =========================================
def run_prophet(daily_df):
    """
    Input:  daily_df with columns [data_date, cost, conversions]
    Output: forecast dataframe + kpi dict
    """
    daily_df = daily_df.sort_values('data_date').copy()
    daily_df['cpl'] = daily_df['cost'] / daily_df['conversions']

    # IQR capping
    Q1    = daily_df['cpl'].quantile(0.25)
    Q3    = daily_df['cpl'].quantile(0.75)
    IQR   = Q3 - Q1
    lower = max(Q1 - 1.5 * IQR, 0.01)
    upper = Q3 + 1.5 * IQR
    daily_df['cpl_capped'] = daily_df['cpl'].clip(lower, upper)

    prophet_df = daily_df[['data_date','cpl_capped']].rename(
        columns={'data_date':'ds','cpl_capped':'y'})
    prophet_df['y'] = np.log(prophet_df['y'])

    model = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=True,
        daily_seasonality=False,
        interval_width=0.8,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=5,
        n_changepoints=10
    )
    model.fit(prophet_df)

    future   = model.make_future_dataframe(periods=7)
    forecast = model.predict(future)
    forecast['yhat']       = np.exp(forecast['yhat'])
    forecast['yhat_lower'] = np.exp(forecast['yhat_lower'])
    forecast['yhat_upper'] = np.exp(forecast['yhat_upper'])

    # MAPE on last 7 days fitted vs actual
    actual_window = daily_df[daily_df['data_date'] >= last7_start]
    forecast_hist = forecast[(forecast['ds'] >= last7_start) &
                              (forecast['ds'] <= reference_date)]
    merged = actual_window[['data_date','cpl']].merge(
        forecast_hist[['ds','yhat']].rename(columns={'ds':'data_date'}),
        on='data_date', how='inner')
    if len(merged) > 0:
        mape = (np.abs(merged['cpl'] - merged['yhat']) / merged['cpl']).mean() * 100
        mape = min(mape, 999)
    else:
        mape = np.nan

    if np.isnan(mape):        reliability = "Unknown"
    elif mape < 15:           reliability = "High"
    elif mape < 35:           reliability = "Medium"
    else:                     reliability = "Low"

    # KPIs — weighted CPL (matches Power BI exactly)
    last7      = daily_df[daily_df['data_date'] >= last7_start]
    prev7      = daily_df[(daily_df['data_date'] >= prev7_start) &
                          (daily_df['data_date'] <= prev7_end)]

    last7_cost = last7['cost'].sum()
    last7_conv = last7['conversions'].sum()
    prev7_cost = prev7['cost'].sum()
    prev7_conv = prev7['conversions'].sum()

    last7_cpl  = last7_cost / last7_conv if last7_conv > 0 else None
    prev7_cpl  = prev7_cost / prev7_conv if prev7_conv > 0 else None

    if last7_cpl and prev7_cpl:
        current_pct = ((last7_cpl - prev7_cpl) / prev7_cpl) * 100
        if last7_cpl < prev7_cpl:   trend = "Improving"
        elif last7_cpl > prev7_cpl: trend = "Declining"
        else:                        trend = "Stable"
    else:
        current_pct = None
        trend       = None

    forecast_future  = forecast[forecast['ds'] > reference_date]
    forecast_avg_cpl = forecast_future['yhat'].mean()

    if last7_cpl:
        fcast_pct = ((forecast_avg_cpl - last7_cpl) / last7_cpl) * 100
        if fcast_pct < -8:   forecast_direction = "Improving"
        elif fcast_pct > 8:  forecast_direction = "Declining"
        else:                 forecast_direction = "Stable"
    else:
        fcast_pct          = None
        forecast_direction = None

    kpis = {
        'last7_cpl':          last7_cpl,
        'prev7_cpl':          prev7_cpl,
        'last7_cost':         last7_cost,
        'last7_conv':         int(last7_conv),
        'trend':              trend,
        'current_pct':        round(current_pct, 2) if current_pct else None,
        'forecast_avg_cpl':   forecast_avg_cpl,
        'fcast_pct':          round(fcast_pct, 2) if fcast_pct else None,
        'forecast_direction': forecast_direction,
        'mape':               round(mape, 2) if not np.isnan(mape) else None,
        'reliability':        reliability,
    }

    return forecast_future, kpis


# =========================================
# FORECASTING — PER CAMPAIGN
# =========================================
results = []
campaigns = df['campaign_name'].unique()

for campaign in campaigns:
    campaign_df = df[df['campaign_name'] == campaign].copy()
    if len(campaign_df) < 14:
        print(f"Skipping {campaign} — not enough data ({len(campaign_df)} days)")
        continue

    print(f"Forecasting: {campaign}")
    try:
        forecast_future, kpis = run_prophet(campaign_df)
    except Exception as e:
        print(f"  Error: {e}")
        continue

    for _, row in forecast_future.iterrows():
        results.append({
            "data_date":          row['ds'],
            "campaign_name":      campaign,
            "actual_cpl":         None,
            "forecast_cpl":       round(row['yhat'], 4),
            "lower_ci":           round(row['yhat_lower'], 4),
            "upper_ci":           round(row['yhat_upper'], 4),
            "last7_cpl":          kpis['last7_cpl'],
            "prev7_cpl":          kpis['prev7_cpl'],
            "trend":              kpis['trend'],
            "current_pct":        kpis['current_pct'],
            "forecast_avg_cpl":   kpis['forecast_avg_cpl'],
            "fcast_pct":          kpis['fcast_pct'],
            "forecast_direction": kpis['forecast_direction'],
            "mape":               kpis['mape'],
            "reliability":        kpis['reliability'],
        })

# =========================================
# FORECASTING — ALL CAMPAIGNS COMBINED
# Aggregates daily cost + conversions first,
# then runs Prophet — matches Power BI script exactly
# =========================================
print("Forecasting: All Campaigns Combined")
portfolio_daily = (df.groupby('data_date')
                     .agg(cost=('cost','sum'), conversions=('conversions','sum'))
                     .reset_index())

try:
    forecast_future, kpis = run_prophet(portfolio_daily)
    for _, row in forecast_future.iterrows():
        results.append({
            "data_date":          row['ds'],
            "campaign_name":      "All Campaigns Combined",
            "actual_cpl":         None,
            "forecast_cpl":       round(row['yhat'], 4),
            "lower_ci":           round(row['yhat_lower'], 4),
            "upper_ci":           round(row['yhat_upper'], 4),
            "last7_cpl":          kpis['last7_cpl'],
            "prev7_cpl":          kpis['prev7_cpl'],
            "trend":              kpis['trend'],
            "current_pct":        kpis['current_pct'],
            "forecast_avg_cpl":   kpis['forecast_avg_cpl'],
            "fcast_pct":          kpis['fcast_pct'],
            "forecast_direction": kpis['forecast_direction'],
            "mape":               kpis['mape'],
            "reliability":        kpis['reliability'],
        })
    print(f"  Portfolio forecast CPL: ${kpis['forecast_avg_cpl']:.2f}")
    print(f"  Portfolio last 7 CPL:   ${kpis['last7_cpl']:.2f}")
    print(f"  Portfolio trend:        {kpis['trend']}")
except Exception as e:
    print(f"  Portfolio forecast error: {e}")

# =========================================
# HISTORICAL DATA — per campaign
# =========================================
actual_table = df[['data_date','campaign_name','cpl']].copy()
actual_table = actual_table.rename(columns={'cpl':'actual_cpl'})
for col in ['forecast_cpl','lower_ci','upper_ci','last7_cpl','prev7_cpl',
            'trend','current_pct','forecast_avg_cpl','fcast_pct',
            'forecast_direction','mape','reliability']:
    actual_table[col] = None

# =========================================
# HISTORICAL DATA — All Campaigns Combined
# =========================================
portfolio_hist = portfolio_daily.copy()
portfolio_hist['campaign_name'] = "All Campaigns Combined"
portfolio_hist['actual_cpl']    = portfolio_hist['cost'] / portfolio_hist['conversions']
for col in ['forecast_cpl','lower_ci','upper_ci','last7_cpl','prev7_cpl',
            'trend','current_pct','forecast_avg_cpl','fcast_pct',
            'forecast_direction','mape','reliability']:
    portfolio_hist[col] = None
portfolio_hist = portfolio_hist.drop(columns=['cost','conversions'])

# =========================================
# COMBINE ALL
# =========================================
forecast_df = pd.DataFrame(results)
final_table = pd.concat([actual_table, portfolio_hist, forecast_df], ignore_index=True)
final_table = final_table.sort_values(['campaign_name','data_date']).reset_index(drop=True)
print("\nFinal dataset:", final_table.shape)

# =========================================
# PREP FOR GOOGLE SHEETS
# =========================================
final_table['data_date'] = final_table['data_date'].astype(str)
final_table = final_table.fillna("")

# =========================================
# GOOGLE SHEETS
# =========================================
scope = [
    'https://spreadsheets.google.com/feeds',
    'https://www.googleapis.com/auth/drive'
]
creds = ServiceAccountCredentials.from_json_keyfile_name(
    'google_credentials.json',
    scope
)
client    = gspread.authorize(creds)
sheet     = client.open_by_url(
    "https://docs.google.com/spreadsheets/d/1ZGfjP3YyOqs5rBlfjfYjQQXU5FBj-xbougW3wlQJGMw"
)
worksheet = sheet.worksheet("Google Ads")
worksheet.clear()
worksheet.update(
    [final_table.columns.tolist()] +
    final_table.values.tolist()
)
print("Google Sheet Updated Successfully")
print(f"Total rows written: {len(final_table)}")

Connected to Redshift


/tmp/ipykernel_537/170300629.py:75: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Rows loaded: (10559, 4)
After cleaning: (516, 5)
Reference date (yesterday): 2026-03-05
Forecasting: PCP_OA_GPT_2_UF_Dgen_Alina
Forecasting: PCP_OA_HSL_GPT_1_DR_Dgen_M
Forecasting: PCP_OA_LJP_GPT_3_MyReclaim_Dgen_M
Forecasting: PCP_OA_GPT_2_UF_Display_Alina
Forecasting: PCP_OA_Scan_And_Claim_GPT_1_Dgen_M
Forecasting: PCP_OA_LJP_GPT_3_UF_Dgen_Alina
Skipping PCP_OA_Scan_And_Claim_GPT_1_PMax_M — not enough data (2 days)
Forecasting: PCP_RHL_LJP_GPT_3_Locksley_Dgen_M
Skipping PCP_RF_LJP_GPT_1_Locksley_Dgen_M — not enough data (10 days)
Forecasting: PCP_RHL_LJP_GPT_3_Locksley_Search_M
Forecasting: PCP_RF_LJP_GPT_2_Locksley_Dgen_M
Forecasting: PCP_Alawco_GPT_2_mycarloanclaims_Dgen_M
Skipping PCP_Alawco_GPT_1_mycarloanclaims_Dgen_M — not enough data (12 days)
Forecasting: PCP_BG_LJP_GPT_1_Benson_Dgen_M
Forecasting: PCP_RF_LJP_GPT_2_Fastpcp_Dgen_M
Skipping OA_PCP_GPT_2_Display_M — not enough data (3 days)
Skipping PCP_RHL_LJP_GPT_3_Locksley_Display_M — not enough data (6 days)
Forecasting: All

/tmp/ipykernel_537/170300629.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_table = pd.concat([actual_table, portfolio_hist, forecast_df], ignore_index=True)


Google Sheet Updated Successfully
Total rows written: 652
